%md
# Notebook: `07. Gold`
---

## Descrição: 

Esse notebook realiza a aplicação das regras de negócio, agregações e joins finais para a construção das tabelas dimensão e fato, para análises finais.

## Origem

- **yellowTaxiTripData**: `mvp.silver.yellowTaxiTripData`;
- **greenTaxiTripData**: `mvp.silver.greenTaxiTripData`;
- **forHireVehicles**: `mvp.silver.forHireVehicles`
- **forHireVehiclesHighVolume**: `mvp.silver.forHireVehiclesHighVolume`
- **taxi_zone_lookup**: `mvp.silver.taxi_zone_lookup`
- **fhv_base_lookup**: `mvp.silver.fhv_base_lookup`
- **Feriados_US_NY**: `mvp.silver.Feriados_US_NY`
- **Inflacao_CPI_U**: `mvp.silver.Inflacao_CPI_U`
- **Inflacao_CPI_T**: `mvp.silver.Inflacao_CPI_T`

## Destino

- **dim_zonas_de_taxi**: `mvp.gold.dim_zonas_de_taxi`;
- **dim_calendario**: `mvp.gold.dim_calendario`;
- **dim_tipo_veiculo**: `mvp.gold.dim_tipo_veiculo`
- **fato_Inflacao_CPI**: `mvp.gold.fato_Inflacao_CPI`
- **fato_corrida_mensal**: `mvp.gold.fato_corrida_mensal`
- **fato_demanda_horaria**: `mvp.gold.fato_demanda_horaria`
- **fato_corridas_suspeitas**: `mvp.gold.fato_corridas_suspeitas`

Define o catálogo e schema usado.

In [0]:
%sql
USE mvp.gold

Importa as bibliotecas usadas.

In [0]:
from pyspark.sql.functions import explode, to_date

### gold.dim_Zonas_de_Taxi (Dimensão de Zonas de Táxi)

Leitura da tabela `mvp.silver.taxi_zone_lookup` e gravação da dimensão `mvp.gold.dim_Zonas_de_Taxi`. Não há filtros nem alteração de conteúdo: a única transformação é a renomeação das colunas para português, seguindo a nomenclatura do modelo dimensional.

| Coluna de origem | Coluna na dimensão |
|---|---|
| `LocationID` | `IDLocal` |
| `Borough` | `Distrito` |
| `Zone` | `Zona` |
| `service_zone` | `Zona_Servico` |

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.gold.dim_Zonas_de_Taxi")
```

In [0]:
df = spark.table("mvp.silver.taxi_zone_lookup")

df_final = (
    df
    .withColumnRenamed("LocationID", "IDLocal")
    .withColumnRenamed("Borough", "Distrito")
    .withColumnRenamed("Zone", "Zona")
    .withColumnRenamed("service_zone", "Zona_Servico")
)

df_final.write.mode("overwrite").saveAsTable("mvp.gold.dim_Zonas_de_Taxi")

num_affected_rows,num_inserted_rows


### gold.dim_Calendario (Dimensão de Calendário)

Geração da dimensão `mvp.gold.dim_Calendario`, com uma linha por dia entre 01/01/2016 e 31/12/2026, cobrindo todo o período dos dados de viagens. A tabela é construída a partir de uma sequência de datas gerada em SQL e enriquecida com a tabela `mvp.silver.feriados`, sendo a base das agregações por ano, mês e tipo de dia (útil ou não útil) nas perguntas 5 e 6.

| Coluna | Descrição |
|---|---|
| `Data` | Data do calendário (chave da dimensão) |
| `Ano` | Ano da data |
| `Nr_Mes` | Número do mês (1 a 12) |
| `Fl_Fim_de_Semana` | `TRUE` para sábados e domingos |
| `Fl_Feriado` | `TRUE` se a data consta na tabela `silver.feriados` (feriados de fato observados) |
| `Fl_Dia_Util` | `TRUE` quando a data não é fim de semana nem feriado |

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.gold.dim_Calendario")
```

In [0]:
data_inicial = '2016-01-01'
data_final = '2026-12-31'


df_datas = spark.sql(
  f"SELECT sequence(to_date('{data_inicial}'), to_date('{data_final}')) AS date_array"
)

df_lista_datas = df_datas.select(explode("date_array").alias("Data"))

df_feriados = spark.table("mvp.silver.feriados")

df_lista_datas.createOrReplaceTempView("df_lista_datas")
df_feriados.createOrReplaceTempView("df_feriados")

df_final = spark.sql("""
WITH CTE_Cal_Base AS (
SELECT
     Data               AS Data,
     YEAR(Data)         AS Ano,
     MONTH(Data)        AS Nr_Mes,
     CASE
        WHEN WEEKDAY(Data) >= 5 THEN TRUE
        ELSE FALSE
     END AS Fl_Fim_de_Semana,
     CASE 
        WHEN Data IN (SELECT date FROM df_feriados) THEN TRUE
        ELSE FALSE 
    END AS Fl_Feriado
FROM df_lista_datas
)

SELECT
      *,
      NOT (Fl_Fim_de_Semana OR Fl_Feriado) AS Fl_Dia_Util
FROM CTE_Cal_Base
""")

df_final.write.mode("overwrite").saveAsTable("mvp.gold.dim_Calendario")

### gold.dim_tipo_veiculo (Dimensão de Tipo de Veículo)

Geração da dimensão `mvp.gold.dim_tipo_veiculo`, que unifica em uma única tabela os tipos de veículo de todas as bases (táxis e serviços de aplicativo). Ela é composta pela união de duas fontes:

1. **Serviços de aplicativo (For Hire Service)**: valores distintos de licença de operadora de alto volume (`High_Volume_License_Number`) e empresa afiliada (`App_Company_Affiliation`) extraídos de `mvp.silver.fhv_base_lookup`, com `Categoria` fixada em "For Hire Service". A licença da operadora é usada como `ID_Veiculo`, o que permite relacionar a dimensão tanto com a base HVFHS (que registra essa licença) quanto com a base FHV legada (cujos códigos de base de despacho são convertidos para a licença correspondente, conforme descrito nas escolhas de modelagem).
2. **Táxis**: duas linhas inseridas manualmente com chaves substitutas, `TA` (Táxi Amarelo) e `TV` (Táxi Verde), com `Categoria` "Taxi".

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.gold.dim_tipo_veiculo")
```

In [0]:
df_fhvlk = spark.table("mvp.silver.fhv_base_lookup")

df_fhvlk.createOrReplaceTempView("df_fhvlk")


df_final = spark.sql("""
WITH CTE_FHVLK_Distinto AS (
    SELECT DISTINCT
        High_Volume_License_Number AS ID_Veiculo,
        'For Hire Service'         AS Categoria,
        App_Company_Affiliation    AS Tipo_Veiculo
    FROM df_fhvlk
)

SELECT * FROM CTE_FHVLK_Distinto

UNION ALL

SELECT * FROM (VALUES
    ('TV', 'Taxi', 'Taxi Verde'),
    ('TA', 'Taxi', 'Taxi Amarelo')
) AS Taxis(ID_Veiculo, Categoria, Tipo_Veiculo)
""")

df_final.write.mode("overwrite").saveAsTable("mvp.gold.dim_tipo_veiculo")

### gold.fato_Inflacao_CPI (Fato de Inflação)

Geração da tabela `mvp.gold.fato_Inflacao_CPI`, que consolida em uma única tabela mensal os dois índices de inflação americana: o CPI geral (`silver.inflacao_cpi_u`) e o CPI de transportes (`silver.inflacao_cpi_t`). A junção é feita pela data de observação (`left join`, tendo o CPI geral como tabela base), e a tabela é a base das comparações de tarifa e remuneração com a inflação nas perguntas 7 e 8.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_Inflacao_CPI")
```

In [0]:
df_CPIU = spark.table("mvp.silver.inflacao_cpi_u")

df_CPIT = spark.table("mvp.silver.inflacao_cpi_t")

df_join = df_CPIU.join(df_CPIT, df_CPIU.observation_date == df_CPIT.observation_date, "left")

df_final = (
    df_join
    .select(
        df_CPIU.observation_date.alias("Data")
        , df_CPIU.CPIAUCSL.alias("Indice_CPI_U")
        , df_CPIT.CPITRNSL.alias("Indice_CPI_T")
    )
)

df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_Inflacao_CPI")

num_affected_rows,num_inserted_rows


### gold.fato_corrida_mensal (Fato de Corridas Mensais)

Geração da tabela `mvp.gold.fato_corrida_mensal`, que agrega as viagens de todos os tipos de veículo ao grão **mês x local de embarque x veículo x método de pagamento**. É a base das perguntas 1, 2, 3, 5 e 7. A tabela é construída pela união (`UNION ALL`) de quatro consultas, uma por fonte da camada silver, todas padronizadas nas mesmas colunas:

| Fonte | `ID_Veiculo` | `Metodo_Pagamento` | Tarifa, distância e remuneração |
|---|---|---|---|
| `silver.yellowtaxitripdata` | `TA` | Preenchido a partir de `payment_type` | Tarifa base e distância; remuneração `NULL` |
| `silver.greentaxitripdata` | `TV` | Preenchido a partir de `payment_type` | Tarifa base e distância; remuneração `NULL` |
| `silver.forhirevehicles` (até jan/2019) | Licença HVFHS da operadora | `NULL` | Apenas contagem de viagens; demais indicadores `NULL` |
| `silver.forhirevehicleshighvolume` (a partir de fev/2019) | Licença HVFHS da operadora | `NULL` | Tarifa base e remuneração; distância não agregada |

**Transformações e regras aplicadas:**

1. **Grão mensal**: a data de embarque é convertida para o último dia do mês (`LAST_DAY`), padronizando a chave de relacionamento com `dim_Calendario`.
2. **Indicadores como soma**: `Qtd_Viagens` (contagem), `Tarifa_Base`, `Distancia_Total` e `Remuneracao_Motorista` são armazenados como somas, permitindo calcular médias e razões em qualquer nível de agregação nas análises finais.
3. **Método de pagamento (táxis)**: o código numérico de `payment_type` é traduzido para descrição (0 Flex Fare, 1 Cartão de crédito, 2 Dinheiro, 3 Sem cobrança, 4 Disputado, 5 Desconhecido, 6 Viagem anulada).
4. **Chave de veículo**: táxis usam as chaves substitutas `TA` e `TV`; serviços de aplicativo usam a licença de operadora de alto volume (`hvfhs_license_num`), relacionando-se com `dim_tipo_veiculo`.
5. **Histórico anterior a 2019 (FHV)**: as viagens da base legada são associadas à operadora por meio da tabela `fhv_base_lookup` (do código de base de despacho para a licença HVFHS) e mantidas somente as operadoras Uber, Lyft, Juno e Via, descartando as demais bases da FHV.
6. **Filtro de operadoras (HVFHS)**: da base de alto volume também são mantidas apenas as operadoras Uber, Lyft, Juno e Via.
7. **Remuneração do motorista**: definida como a soma de `driver_pay` e `tips`, ou seja, o ganho total do motorista incluindo gorjetas.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_corrida_mensal")
```

In [0]:
df_yellow = spark.table("mvp.silver.yellowtaxitripdata")
df_green = spark.table("mvp.silver.greentaxitripdata")
df_fhv = spark.table("mvp.silver.forhirevehicles")
df_fhvhv = spark.table("mvp.silver.forhirevehicleshighvolume")
df_fhvlk = spark.table("mvp.silver.fhv_base_lookup")

df_yellow.createOrReplaceTempView("df_yellow")
df_green.createOrReplaceTempView("df_green")
df_fhv.createOrReplaceTempView("df_fhv")
df_fhvhv.createOrReplaceTempView("df_fhvhv")
df_fhvlk.createOrReplaceTempView("df_fhvlk")


df_final = spark.sql("""
WITH CTE_FHVLK_Distinto AS (
    SELECT DISTINCT
        High_Volume_License_Number,
        App_Company_Affiliation
    FROM df_fhvlk
),
CTE_Amarelo AS (
    SELECT
        LAST_DAY(tpep_pickup_date)   AS Data,
        'TA'                         AS ID_Veiculo,
        PULocationID                 AS ID_Embarque,
        CASE payment_type
            WHEN 0 THEN 'Flex Fare'
            WHEN 1 THEN 'Cartão de crédito'
            WHEN 2 THEN 'Dinheiro'
            WHEN 3 THEN 'Sem cobrança'
            WHEN 4 THEN 'Disputado'
            WHEN 5 THEN 'Desconhecido'
            WHEN 6 THEN 'Viagem anulada'
        END                                                 AS Metodo_Pagamento,
        COUNT(*)                                            AS Qtd_Viagens,
        CAST(SUM(fare_amount) AS Decimal(12,2))             AS Tarifa_Base,
        CAST(SUM(trip_distance) AS Decimal(12,2))           AS Distancia_Total,
        NULL                                                AS Remuneracao_Motorista
    FROM df_yellow

    GROUP BY LAST_DAY(tpep_pickup_date), PULocationID, payment_type
),

CTE_Verde AS (
    SELECT
        LAST_DAY(lpep_pickup_date)   AS Data,
        'TV'                         AS ID_Veiculo,
        PULocationID                 AS ID_Embarque,
        CASE payment_type
            WHEN 0 THEN 'Flex Fare'
            WHEN 1 THEN 'Cartão de crédito'
            WHEN 2 THEN 'Dinheiro'
            WHEN 3 THEN 'Sem cobrança'
            WHEN 4 THEN 'Disputado'
            WHEN 5 THEN 'Desconhecido'
            WHEN 6 THEN 'Viagem anulada'
        END                                                   AS Metodo_Pagamento,
        COUNT(*)                                              AS Qtd_Viagens,
        CAST(SUM(fare_amount) AS Decimal(12, 2))              AS Tarifa_Base,
        CAST(SUM(trip_distance) AS Decimal(12,2))             AS Distancia_Total,
        NULL                                                  AS Remuneracao_Motorista
    FROM df_green

    GROUP BY LAST_DAY(lpep_pickup_date), PULocationID, payment_type
),

CTE_FHV AS (
    SELECT
        LAST_DAY(f.pickup_date)       AS Data,
        lk.High_Volume_License_Number AS ID_Veiculo,
        PULocationID                  AS ID_Embarque,
        NULL                          AS Metodo_Pagamento,
        COUNT(*)                      AS Qtd_Viagens,
        NULL                          AS Tarifa_Base,
        NULL                          AS Distancia_Total,
        NULL                          AS Remuneracao_Motorista
    FROM df_fhv f
    LEFT JOIN df_fhvlk lk
        ON lk.License_Number = f.dispatching_base_num
    
    WHERE lk.App_Company_Affiliation IN ('Uber', 'Lyft', 'Juno', 'Via')

    GROUP BY LAST_DAY(f.pickup_date), PULocationID, lk.High_Volume_License_Number
),

CTE_FHVHV AS (
    SELECT
        LAST_DAY(f.pickup_date)       AS Data,
        f.hvfhs_license_num           AS ID_Veiculo,
        f.PULocationID                AS ID_Embarque,
        NULL                          AS Metodo_Pagamento,
        COUNT(*)                      AS Qtd_Viagens,
        CAST(SUM(f.base_passenger_fare) AS Decimal(12,2))    AS Tarifa_Base,
        CAST(SUM(f.driver_pay + f.tips) AS Decimal(12,2))    AS Remuneracao_Motorista
    FROM df_fhvhv f
    LEFT JOIN CTE_FHVLK_Distinto lk
        ON lk.High_Volume_License_Number = f.hvfhs_license_num
    
    WHERE lk.App_Company_Affiliation IN ('Uber', 'Lyft', 'Juno', 'Via')

    GROUP BY LAST_DAY(f.pickup_date), f.PUlocationID, f.hvfhs_license_num
)

SELECT * FROM CTE_Amarelo

UNION ALL

SELECT * FROM CTE_Verde

UNION ALL 

SELECT * FROM CTE_FHV

UNION ALL

SELECT * FROM CTE_FHVHV
""")

df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_corrida_mensal")

### gold.fato_demanda_horaria (Fato de Demanda Horária)

Geração da tabela `mvp.gold.fato_demanda_horaria`, que agrega as viagens de todos os tipos de veículo ao grão **dia de embarque x dia de desembarque x hora de embarque x hora de desembarque x local de embarque x local de desembarque x veículo**. É a base das perguntas 5 e 6 (bairros e horários de pico de embarque e desembarque em dias úteis e não úteis). A tabela é construída pela união (`UNION ALL`) de quatro consultas, uma por fonte da camada silver, todas padronizadas nas mesmas colunas:

| Fonte | `ID_Veiculo` | Observação |
|---|---|---|
| `silver.yellowtaxitripdata` | `TA` | Táxi amarelo |
| `silver.greentaxitripdata` | `TV` | Táxi verde |
| `silver.forhirevehicles` (até jan/2019) | Licença HVFHS da operadora | Histórico anterior à base HVFHS, associado à operadora via `fhv_base_lookup` |
| `silver.forhirevehicleshighvolume` (a partir de fev/2019) | Licença HVFHS da operadora | Base de alto volume |

| Coluna | Descrição |
|---|---|
| `Data_Embarque` | Data de embarque (relaciona-se com `dim_Calendario`) |
| `Data_Destino` | Data de desembarque (relaciona-se com `dim_Calendario`) |
| `Hora_Embarque` | Hora inteira do embarque (0 a 23) |
| `Hora_Destino` | Hora inteira do desembarque (0 a 23) |
| `ID_Veiculo` | Chave do tipo de veículo (relaciona-se com `dim_tipo_veiculo`) |
| `ID_Embarque` | Zona de embarque (relaciona-se com `dim_Zonas_de_Taxi`) |
| `ID_Destino` | Zona de desembarque (relaciona-se com `dim_Zonas_de_Taxi`) |
| `Qtd_Viagens` | Número de viagens no grão da linha |

**Transformações e regras aplicadas:**

1. **Agregação por contagem**: cada linha representa a quantidade de viagens (`COUNT(*)`) para uma combinação de datas, horas, zonas e veículo. Não há indicadores monetários nesta tabela, apenas volume.
2. **Chave de veículo**: táxis usam as chaves substitutas `TA` e `TV`; serviços de aplicativo usam a licença de operadora de alto volume.
3. **Histórico anterior a 2019 (FHV)**: as viagens da base legada são associadas à operadora por meio da tabela `fhv_base_lookup` (do código de base de despacho para a licença HVFHS), mantendo somente as operadoras Uber, Lyft, Juno e Via.
4. **Filtro de operadoras (HVFHS)**: da base de alto volume também são mantidas apenas as operadoras Uber, Lyft, Juno e Via.

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_demanda_horaria")
```

In [0]:
df_yellow = spark.table("mvp.silver.yellowtaxitripdata")
df_green = spark.table("mvp.silver.greentaxitripdata")
df_fhv = spark.table("mvp.silver.forhirevehicles")
df_fhvhv = spark.table("mvp.silver.forhirevehicleshighvolume")
df_fhvlk = spark.table("mvp.silver.fhv_base_lookup")

df_yellow.createOrReplaceTempView("df_yellow")
df_green.createOrReplaceTempView("df_green")
df_fhv.createOrReplaceTempView("df_fhv")
df_fhvhv.createOrReplaceTempView("df_fhvhv")
df_fhvlk.createOrReplaceTempView("df_fhvlk")


df_final = spark.sql("""
WITH CTE_FHVLK_Distinto AS (
    SELECT DISTINCT
        High_Volume_License_Number,
        App_Company_Affiliation
    FROM df_fhvlk
),
CTE_Amarelo AS (
    SELECT
        tpep_pickup_date             AS Data_Embarque,
        tpep_dropoff_date            AS Data_Destino,
        pickup_hour                  AS Hora_Embarque,
        dropoff_hour                 AS Hora_Destino,
        'TA'                         AS ID_Veiculo,
        PULocationID                 AS ID_Embarque,
        DOLocationID                 AS ID_Destino,
        COUNT(*)                     AS Qtd_Viagens
    FROM df_yellow

    GROUP BY tpep_pickup_date, tpep_dropoff_date, pickup_hour, dropoff_hour, PULocationID, DOLocationID
),

CTE_Verde AS (
    SELECT
        lpep_pickup_date              AS Data_Embarque,
        lpep_dropoff_date             AS Data_Destino,
        pickup_hour                   AS Hora_Embarque,
        dropoff_hour                  AS Hora_Destino,
        'TV'                          AS ID_Veiculo,
        PULocationID                  AS ID_Embarque,
        DOLocationID                  AS ID_Destino,
        COUNT(*)                      AS Qtd_Viagens
    FROM df_green

    GROUP BY lpep_pickup_date, lpep_dropoff_date, pickup_hour, dropoff_hour, PULocationID, DOLocationID
),

CTE_FHV AS (
    SELECT
        f.pickup_date                 AS Data_Embarque,
        f.dropOff_date                AS Data_Destino,
        f.pickup_hour                 AS Hora_Embarque,
        f.dropoff_hour                AS Hora_Destino,
        lk.High_Volume_License_Number AS ID_Veiculo,
        f.PULocationID                AS ID_Embarque,
        f.DOLocationID                AS ID_Destino,
        COUNT(*)                      AS Qtd_Viagens
    FROM df_fhv f
    LEFT JOIN df_fhvlk lk
        ON lk.License_Number = f.dispatching_base_num
    
    WHERE lk.App_Company_Affiliation IN ('Uber', 'Lyft', 'Juno', 'Via')

    GROUP BY f.pickup_date, f.dropOff_date, f.pickup_hour, f.dropoff_hour, 
        lk.High_Volume_License_Number, f.PULocationID, f.DOLocationID
),

CTE_FHVHV AS (
    SELECT
        f.pickup_date                 AS Data_Embarque,
        f.dropoff_date                AS Data_Destino,
        f.pickup_hour                 AS Hora_Embarque,
        f.dropoff_hour                AS Hora_Destino,
        f.hvfhs_license_num           AS ID_Veiculo,
        f.PULocationID                AS ID_Embarque,
        f.DOLocationID                AS ID_Destino,
        COUNT(*)                      AS Qtd_Viagens
    FROM df_fhvhv f
    LEFT JOIN CTE_FHVLK_Distinto lk
        ON lk.High_Volume_License_Number = f.hvfhs_license_num
    
    WHERE lk.App_Company_Affiliation IN ('Uber', 'Lyft', 'Juno', 'Via')

    GROUP BY f.pickup_date, f.dropOff_date, f.pickup_hour, f.dropoff_hour, f.hvfhs_license_num, f.PULocationID, f.DOLocationID
)

SELECT * FROM CTE_Amarelo

UNION ALL

SELECT * FROM CTE_Verde

UNION ALL 

SELECT * FROM CTE_FHV

UNION ALL

SELECT * FROM CTE_FHVHV
""")

df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_demanda_horaria")

### gold.fato_corridas_suspeitas (Fato de Corridas Suspeitas)

Geração da tabela `mvp.gold.fato_corridas_suspeitas`, que lista, no **grão individual de viagem** (sem agregação), as corridas de táxis amarelos e verdes que violam as regras de operação descritas na seção de Contexto de Negócio. É a base da pergunta 4. Apenas táxis amarelos e verdes são avaliados, pois as restrições geográficas não se aplicam aos serviços de aplicativo. A tabela é construída pela união (`UNION ALL`) de duas consultas, uma por tipo de táxi, a partir das tabelas `silver.yellowtaxitripdata`, `silver.greentaxitripdata` e da tabela de zonas `silver.taxi_zone_lookup`.

**Regras de irregularidade aplicadas:**

| Tipo de táxi | Regra | `Motivo_Irregularidade` |
|---|---|---|
| Amarelo e verde | Embarque fora da cidade de Nova Iorque (`PULocationID` = 265) | Embarque Fora de Nova Iorque |
| Verde | Embarque em aeroporto (`service_zone` = "Airports") sem tarifa negociada (`RatecodeID` diferente de 5) | Embarque em Aeroporto não negociado |
| Verde | Embarque em zona exclusiva de táxis amarelos (`service_zone` = "Yellow Zone") | Embarque em zona de taxis amarelos |

Apenas as viagens que violam ao menos uma regra são carregadas; as demais são descartadas.

| Coluna | Descrição |
|---|---|
| `Provedor` | Fornecedor do sistema de bordo, traduzido a partir do `VendorID` (Creative Mobile Technologies, Curb Mobility, Myle Technologies e Helix, este último apenas no amarelo); códigos não mapeados aparecem como "Desconhecido" |
| `Data_Embarque` / `Data_Destino` | Datas de embarque e desembarque |
| `Hora_Embarque` / `Hora_Destino` | Horas inteiras de embarque e desembarque |
| `ID_Veiculo` | `TA` (amarelo) ou `TV` (verde) |
| `ID_Embarque` / `ID_Destino` | Zonas de embarque e desembarque |
| `Metodo_Pagamento` | Descrição do `payment_type` (Flex Fare, Cartão de crédito, Dinheiro, Sem cobrança, Disputado, Desconhecido, Viagem anulada) |
| `Tipo_Tarifa` | Descrição do `RatecodeID` (Taxa padrão, JFK, Newark, Nassau ou Westchester, Taxa negociada, Viagem em grupo, Nulo/Desconhecido) |
| `Motivo_Irregularidade` | Regra violada pela viagem |

**Comando final de escrita:**

```python
df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_corridas_suspeitas")
```

In [0]:
df_yellow = spark.table("mvp.silver.yellowtaxitripdata")
df_green = spark.table("mvp.silver.greentaxitripdata")
df_tzlk = spark.table("mvp.silver.taxi_zone_lookup")

df_yellow.createOrReplaceTempView("df_yellow")
df_green.createOrReplaceTempView("df_green")
df_tzlk.createOrReplaceTempView("df_tzlk")

df_final = spark.sql("""
WITH CTE_Amarelo AS (
    SELECT
        CASE VendorID
            WHEN 1 THEN 'Creative Mobile Technologies, LLC'
            WHEN 2 THEN 'Curb Mobility, LLC'
            WHEN 6 THEN 'Myle Technologies Inc'
            WHEN 7 THEN 'Helix'
        ELSE 'Desconhecido'
        END                          AS Provedor,
        tpep_pickup_date             AS Data_Embarque,
        tpep_dropoff_date            AS Data_Destino,
        pickup_hour                  AS Hora_Embarque,
        dropoff_hour                 AS Hora_Destino,
        'TA'                         AS ID_Veiculo,
        PULocationID                 AS ID_Embarque,
        DOLocationID                 AS ID_Destino,
        CASE payment_type
            WHEN 0 THEN 'Flex Fare'
            WHEN 1 THEN 'Cartão de crédito'
            WHEN 2 THEN 'Dinheiro'
            WHEN 3 THEN 'Sem cobrança'
            WHEN 4 THEN 'Disputado'
            WHEN 5 THEN 'Desconhecido'
            WHEN 6 THEN 'Viagem anulada'
        END                          AS Metodo_Pagamento,
        CASE RatecodeID
            WHEN 1 THEN  'Taxa padrão'
            WHEN 2 THEN  'JFK'
            WHEN 3 THEN  'Newark'
            WHEN 4 THEN  'Nassau ou Westchester'
            WHEN 5 THEN  'Taxa negociada'
            WHEN 6 THEN  'Viagem em grupo'
            WHEN 99 THEN 'Nulo/Desconhecido'
        END                             AS Tipo_Tarifa,
        'Embarque Fora de Nova Iorque'  AS Motivo_Irregularidade
    FROM df_yellow

    WHERE PULocationID = 265
),

CTE_Verde AS (
    SELECT
        CASE g.VendorID
            WHEN 1 THEN 'Creative Mobile Technologies, LLC'
            WHEN 2 THEN 'Curb Mobility, LLC'
            WHEN 6 THEN 'Myle Technologies Inc'
        ELSE 'Desconhecido'
        END                            AS Provedor,
        g.lpep_pickup_date             AS Data_Embarque,
        g.lpep_dropoff_date            AS Data_Destino,
        g.pickup_hour                  AS Hora_Embarque,
        g.dropoff_hour                 AS Hora_Destino,
        'TV'                           AS ID_Veiculo,
        g.PULocationID                 AS ID_Embarque,
        g.DOLocationID                 AS ID_Destino,
        CASE g.payment_type
            WHEN 0 THEN 'Flex Fare'
            WHEN 1 THEN 'Cartão de crédito'
            WHEN 2 THEN 'Dinheiro'
            WHEN 3 THEN 'Sem cobrança'
            WHEN 4 THEN 'Disputado'
            WHEN 5 THEN 'Desconhecido'
            WHEN 6 THEN 'Viagem anulada'
        END                            AS Metodo_Pagamento,
        CASE g.RatecodeID
            WHEN 1 THEN  'Taxa padrão'
            WHEN 2 THEN  'JFK'
            WHEN 3 THEN  'Newark'
            WHEN 4 THEN  'Nassau ou Westchester'
            WHEN 5 THEN  'Taxa negociada'
            WHEN 6 THEN  'Viagem em grupo'
            WHEN 99 THEN 'Nulo/Desconhecido'
        END                             AS Tipo_Tarifa,
        CASE
            WHEN g.PULocationID = 265 THEN 'Embarque Fora de Nova Iorque'
            WHEN lk.service_zone = 'Airports' AND g.RatecodeID <> 5 THEN 'Embarque em Aeroporto não negociado'
            WHEN lk.service_zone = 'Yellow Zone' THEN 'Embarque em zona de taxis amarelos'
        END                             AS Motivo_Irregularidade
    FROM df_green g

    LEFT JOIN df_tzlk lk
        ON g.PULocationID = lk.LocationID

    WHERE 
    g.PULocationID = 265
    OR (lk.service_zone = 'Airports' AND g.RatecodeID <> 5)
    OR lk.service_zone = 'Yellow Zone'
)

SELECT * FROM CTE_Amarelo

UNION ALL

SELECT * FROM CTE_Verde
""")

df_final.write.mode("overwrite").saveAsTable("mvp.gold.fato_corridas_suspeitas")